In [108]:
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
from dotenv import load_dotenv
load_dotenv()

True

# tool creation

In [109]:

@tool
def get_conversion_factor(base_currency : str, target_currency : str) -> float:
    
    """
    This function fetches the currency conversion factor between a given base currency and target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/99e78127e062abaaf97eb3f8/pair/{base_currency}/{target_currency}'
    response = requests.get(url)

    return response.json()


In [110]:
@tool
def convert(base_currency_value: int, conversion_cost: Annotated[float,InjectedToolArg]):
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """

    return base_currency_value* conversion_cost

In [111]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_cost': {'title': 'Conversion Cost', 'type': 'number'}}

In [112]:
result = get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})
result['conversion_rate']

90.8652

In [113]:
response = convert.invoke({'base_currency_value':12,'conversion_cost':90.9715})
response

1091.6580000000001

# tools binding

In [114]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [115]:
llm_with_tools

RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x00000216E8E5FA90>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_conversion_factor', 'description': 'This function fetches the currency conversion factor between a given base currency and target currency', 'parameters': {'properties': {'base_currency': {'type': 'string'}, 'target_currency': {'type': 'string'}}, 'required': ['base_currency', 'target_currency'], 'type': 'object'}}}, {'type': 'function', 'fun

# tool calling

In [116]:
messages = [HumanMessage('What is conversion factor between USD and INR, and based on that convert the 10 USD into INR')]
messages

[HumanMessage(content='What is conversion factor between USD and INR, and based on that convert the 10 USD into INR', additional_kwargs={}, response_metadata={})]

In [117]:
ai_message = llm_with_tools.invoke(messages)

In [118]:
messages.append(ai_message)

In [119]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': '11133c05-c5ad-41ff-9f59-f768998c8b57',
  'type': 'tool_call'}]

In [120]:
import json

for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        messages.append(tool_message1)
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)

In [121]:
llm_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}, '__gemini_function_call_thought_signatures__': {'da69b0c8-44f5-488f-a57b-30b0aa36c271': 'CtUEAb4+9vsBdMYoaoVBaipGQ94/kakSUgmSjoxe7I3uIeAIJeLBN5YfwgIY3q6+jkDyDjgUfsymKUdBhsZxZpdZrxwklTxH3d+nT3Pyzk3hh+gpSXiW5Cgq0l0W3wvTNrndeysnaG3j6u4McqD1VPV/wkawe/Xc+Go79eigJLdq4uNgzH1M6vXpgHeOFu5JFSI0EBDPDXQdTtlcLEFESm5r0aW5FUS6ofCkeceMmOlRasIozXavzpj3H5BTezRJvt3RvtE2wlMmvVmJir8mTDojumw4Drj8Z2jOt6WXcAPbas/GOMJC2xf8KF+JOod0NWx4D6GlzgNCs0N5c9XnmNMQ3EdHsI6Jr4iavpdmf/knPdeAZp6hux+1brgQ3XBqxc9iUUr4Er5plp0sAeuT7h8JtRWnfu/IlaVkqD5M2Jkrn9iLms7jIDYpKIAotclywPdpg3vG0/0JBLcWCLVtKyMG4Nwvij1OlfefxhUBqecK59m8851qPqYHCfpxU8/HTzV6PJ3iBkbiShVTufJ8hSRbsdXvGPPRbV0xGY+/uc42jeexZEpWTGvIAXWLggn4l1RO0y3MrtJdDxPb575A7/0aJTKL33wYL9LyXqZIFJ6N3eQTKLOAoPH3pVsohiMKOtE+QkuDwHXzICW9gyppQ4O4TryWvHG+wadc6n3cOHUH2NjEI9qBhzEOW05zG6Y0YI8eD8rBbRmeheGdEuSMnLl+Hqvxvxjn2NMsWlyVEIMwGYfsZGaJAN8rKutt3WeKpHBHJH0UzvLmrLsIS/g+